# Adaptive STGT campaign: STGT, Adaptive STGT vs Adaptive Fusion GAT-LSTM

This notebook is intentionally gated. It uses the approved workflow only: WSL/GitHub for source, Google Drive for data/artifacts, and standard Google Colab for compute. It does not use a Colab API, MCP server, browser automation, or unattended authorization.

**Large preprocessing requires typing PREPROCESS; training requires typing RUN after preflight. Final test evaluation separately requires typing TEST and freezing one completed run per model and seed.**

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = 'https://github.com/Akifali-1/research.git'
REPO_BRANCH = 'research/initial-pipeline'
PROJECT = Path('/content/research')
MAX_GPU_RUNS = 10
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT)], check=True)
%cd /content/research
local_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote_line = subprocess.check_output(['git', 'ls-remote', 'origin', f'refs/heads/{REPO_BRANCH}'], text=True).strip()
if not remote_line or remote_line.split()[0] != local_commit:
    raise RuntimeError('Repository push gate failed: local HEAD does not match origin/HEAD')
print('Repository commit:', local_commit)

## Interactive Google Drive authorization

Run this cell interactively. The notebook cannot continue until Google Drive is mounted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/supply_chain_research')
CAMPAIGN_NAME = 'adaptive_stgt_campaign'
RAW_DIR = DRIVE_ROOT / 'data' / 'raw'
SOURCE_ZIP = None  # or Path('/content/drive/MyDrive/favorita-grocery-sales-forecasting.zip')
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
ARTIFACT_ROOT = DRIVE_ROOT / 'experiments' / CAMPAIGN_NAME
REGISTRY_PATH = DRIVE_ROOT / f'experiment_registry_{CAMPAIGN_NAME}.json'
for path in [RAW_DIR, PROCESSED_DIR, ARTIFACT_ROOT]: path.mkdir(parents=True, exist_ok=True)
print('Drive authorized at:', DRIVE_ROOT)

In [ ]:
%pip install -q -r requirements.txt
import torch, numpy, pandas, sklearn, yaml
print('torch:', torch.__version__)
print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)
print('sklearn:', sklearn.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('No GPU; training is blocked by the controller. Select a GPU runtime to continue.')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)

In [ ]:
sys.path.insert(0, str(PROJECT))
from orchestrator import prepare_data, validate_archives, validate_pushed_repository, runtime_info
from src.raw_data import prepare_raw_sources
from src.experiment_registry import Registry
print('Configured RAW_DIR:', RAW_DIR)
print('Present files:', sorted(p.name for p in RAW_DIR.iterdir()) if RAW_DIR.is_dir() else '(folder missing)')
RAW_DIR = prepare_raw_sources(RAW_DIR, zip_path=SOURCE_ZIP)
print('Resolved RAW_DIR:', RAW_DIR)
manifest = validate_archives(RAW_DIR)
print('Validated all source CSV schemas and SHA256 hashes:', len(manifest['files']))
print('Full archive integrity is checked by extraction during preprocessing, before any training.')
validate_pushed_repository(PROJECT, REPO_BRANCH)
print(json.dumps(runtime_info(), indent=2))
registry = Registry(REGISTRY_PATH, max_runs=MAX_GPU_RUNS, max_retries=2)
status = registry.status()
print('Budget:', status)
if status['active_experiments']:
    recovery = input('Confirm no prior Colab process is still running; type RECOVER: ').strip()
    if recovery != 'RECOVER':
        raise RuntimeError('Recovery not confirmed; stop rather than risk concurrent GPU runs')
    print('Marked stale active records interrupted:', registry.mark_interrupted_active('user-confirmed inactive prior session'))
if registry.status()['remaining_runs'] <= 0:
    print('Budget exhausted: completed runs may be inspected/evaluated, but no new training can be submitted.')

## Optional preprocessing gate

The previous cell verifies the pushed commit, interactive Drive authorization, all Favorita archives, and remaining budget. The next cell is the only place that may start the large preprocessing step. It does not consume a GPU experiment slot.

In [ ]:
PROCESSED_BASE = DRIVE_ROOT / 'data' / 'processed'
preprocess_confirmation = input('Type PREPROCESS to run or verify processed data on Drive: ').strip()
if preprocess_confirmation == 'PREPROCESS':
    PROCESSED_DIR = prepare_data(PROJECT, PROJECT / 'configs' / 'experiment.yaml', RAW_DIR, PROCESSED_BASE, branch=REPO_BRANCH)
    print('Processed data version:', PROCESSED_DIR)
else:
    raise RuntimeError('Preprocessing gate not confirmed; no preprocessing was started')

In [ ]:
import yaml
config = yaml.safe_load((PROJECT / 'configs' / 'experiment.yaml').read_text())
config['data']['processed_dir'] = str(PROCESSED_DIR)
config['outputs']['checkpoint_dir'] = str(ARTIFACT_ROOT / 'checkpoints')
config['outputs']['predictions_dir'] = str(ARTIFACT_ROOT / 'predictions')
config['outputs']['metrics_dir'] = str(ARTIFACT_ROOT / 'metrics')
config['outputs']['logs_dir'] = str(ARTIFACT_ROOT / 'logs')
config['outputs']['drive_artifact_root'] = str(ARTIFACT_ROOT)
COLAB_CONFIG = Path('/content/experiment_colab.yaml')
COLAB_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(COLAB_CONFIG.read_text())

In [ ]:
retry_confirmation = input('If failed/interrupted attempts exist, type RETRY to resume; type RESTART only if no epoch checkpoint exists; otherwise Enter: ').strip()
confirmation = input('Type RUN only after reviewing repository, Drive, processed data, runtime, and budget checks: ').strip()
if confirmation != 'RUN':
    raise RuntimeError('Training gate not confirmed; no GPU experiment was submitted')
command = [sys.executable, 'orchestrator.py', '--project-root', str(PROJECT), '--branch', REPO_BRANCH, '--config', str(COLAB_CONFIG), '--raw-dir', str(RAW_DIR), '--registry', str(REGISTRY_PATH), '--models', 'stgt', 'adaptive_stgt', 'gat_lstm', '--max-runs', str(MAX_GPU_RUNS), '--allow-execution']
if retry_confirmation in {'RETRY', 'RESTART'}: command.append('--force-retry')
if retry_confirmation == 'RESTART': command.append('--restart-without-checkpoint')
print('Sequential controller command:', ' '.join(command))
subprocess.run(command, cwd=PROJECT, check=True)

## Optional frozen final-test evaluation
Review validation metrics first. Select exactly one completed run for each of the three models and each configured seed. This step freezes the selection and is not part of hyperparameter search. Rerunning completed training skips it; interrupted training resumes only with explicit RETRY and consumes another budget slot.

In [ ]:
records = registry.snapshot()['experiments']
for record in records:
    print(record['experiment_id'], record['status'], record.get('report', ''))
test_confirmation = input('Type TEST only after selecting configurations using validation alone; Enter to stop: ').strip()
if test_confirmation == 'TEST':
    selected_ids = input('Completed IDs (all 3 models × configured seeds, space-separated): ').split()
    evaluation_resume = input('If final evaluation failed, type EVAL_RESUME to resume the same frozen IDs; otherwise Enter: ').strip()
    evaluator_upgrade = input('If these runs use an older commit, type COMPATIBLE_UPGRADE to authorize a checked evaluation-only upgrade; otherwise Enter: ').strip()
    command = [sys.executable, 'orchestrator.py', '--project-root', str(PROJECT), '--branch', REPO_BRANCH, '--config', str(COLAB_CONFIG), '--raw-dir', str(RAW_DIR), '--registry', str(REGISTRY_PATH), '--finalize-test', '--experiment-ids', *selected_ids, '--allow-execution']
    if evaluation_resume == 'EVAL_RESUME': command.append('--resume-final-test')
    if evaluator_upgrade == 'COMPATIBLE_UPGRADE': command.append('--allow-evaluator-upgrade')
    subprocess.run(command, cwd=PROJECT, check=True)
    report = ARTIFACT_ROOT / 'final_test' / 'stages' / 'report' / 'model_comparison.csv'
    if not report.exists(): report = ARTIFACT_ROOT / 'final_test' / 'metrics' / 'model_comparison.csv'
    print(report.read_text())
else:
    print('Test set remains untouched.')

In [ ]:
print(json.dumps(registry.status(), indent=2))
print('Experiment registry:', REGISTRY_PATH)